# Solar Panel Fault Classification
## Deep Learning Project — EfficientNet-B0 + Transfer Learning (v2 — Overfitting Fixed)
**Task:** Multi-class image classification (12 fault categories)  
**Framework:** PyTorch  
**Architecture:** EfficientNet-B0 pretrained on ImageNet, fine-tuned for fault detection  
**Changes in v2:** Stronger dropout, higher weight decay, removed WeightedRandomSampler, added early stopping


## 1. Setup & Imports

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# ── Device ──
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Configuration

In [ ]:
# ── EDIT THESE TWO LINES ──────────────────────────────────────────────────
IMAGE_DIR  = r"C:\Users\roman\DeepLearning-2026\SOLAR_PANELS_FAULT_NN\images"   # folder containing the .jpg files
CSV_PATH   = r"anomaly_classes.csv"
# ─────────────────────────────────────────────────────────────────────────

# Training hyperparameters
IMG_SIZE      = 224
BATCH_SIZE    = 32
NUM_EPOCHS    = 30          # more epochs — early stopping will cut it short
LR            = 1e-3        # head-only phase
LR_FINETUNE   = 1e-4        # full fine-tune phase
WEIGHT_DECAY  = 1e-3        # increased from 1e-4 to fight overfitting
FREEZE_EPOCHS = 5           # epochs to train head only before unfreezing backbone
PATIENCE      = 5           # early stopping patience

## 3. Dataset & Class Distribution

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Total samples: {len(df)}")

counts = df["anomaly_class"].value_counts()
print("\nClass distribution:")
print(counts.to_string())

plt.figure(figsize=(12, 4))
counts.plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Class Distribution — Solar Panel Faults")
plt.ylabel("Sample Count")
plt.xlabel("Anomaly Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Label Encoding & Train/Val/Test Split

In [ ]:
classes     = sorted(df["anomaly_class"].unique())
NUM_CLASSES = len(classes)
label2idx   = {c: i for i, c in enumerate(classes)}
idx2label   = {i: c for c, i in label2idx.items()}

df["label"] = df["anomaly_class"].map(label2idx)
print(f"Number of classes: {NUM_CLASSES}")
print("Label mapping:", label2idx)

# Stratified split: 70% train / 15% val / 15% test
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=SEED)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=SEED)

print(f"\nSplit sizes — Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 5. Imbalance Handling — Class Weights Only
> **v2 change:** Removed `WeightedRandomSampler` — it was causing the model to overfit on duplicated rare-class images. Class weights in the loss function are sufficient and more stable.

In [ ]:
class_counts = Counter(train_df["anomaly_class"])
total        = len(train_df)

class_weights = torch.tensor(
    [total / (NUM_CLASSES * class_counts[cls]) for cls in classes],
    dtype=torch.float32
).to(device)

print("Class weights:")
for cls, w in zip(classes, class_weights.tolist()):
    print(f"  {cls:<20} {w:.4f}")

## 6. Data Augmentation & DataLoaders

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class SolarDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(self.image_dir / row["filename"]).convert("RGB")
        label = int(row["label"])
        if self.transform:
            img = self.transform(img)
        return img, label


train_dataset = SolarDataset(train_df, IMAGE_DIR, train_transforms)
val_dataset   = SolarDataset(val_df,   IMAGE_DIR, eval_transforms)
test_dataset  = SolarDataset(test_df,  IMAGE_DIR, eval_transforms)

# NOTE: shuffle=True, no sampler — fixes overfitting from v1
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=True, persistent_workers=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, persistent_workers=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=True, persistent_workers=False)

print(f"Batches — Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 7. Model — EfficientNet-B0 with Custom Classifier Head
> **v2 change:** Dropout increased from 0.3 → 0.5 to reduce overfitting.

In [ ]:
model = models.efficientnet_b0(weights="IMAGENET1K_V1")

# Freeze all backbone layers initially
for param in model.parameters():
    param.requires_grad = False

# Replace classifier head — dropout increased to 0.5
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=True),    # increased from 0.3
    nn.Linear(in_features, NUM_CLASSES)
)

model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}  (head only — backbone frozen)")

## 8. Training & Evaluation Helpers

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

def get_optimizer(model, lr):
    return optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=WEIGHT_DECAY
    )

def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * imgs.size(0)
            correct    += (outputs.argmax(1) == labels).sum().item()
            total      += imgs.size(0)
    return total_loss / total, correct / total

## 9. Training Loop with Early Stopping
> **v2 changes:** Added early stopping (patience=5). Increased NUM_EPOCHS to 30 — early stopping will cut training short automatically when val loss stops improving.

In [ ]:
# Clear GPU memory before training
import gc
torch.cuda.empty_cache()
gc.collect()

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss  = float("inf")
best_model_path = "best_model.pth"
no_improve     = 0

optimizer = get_optimizer(model, LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Phase switch: unfreeze backbone after FREEZE_EPOCHS ──
    if epoch == FREEZE_EPOCHS + 1:
        print(f"\n[Epoch {epoch}] Unfreezing backbone — switching to fine-tune LR={LR_FINETUNE}")
        for param in model.parameters():
            param.requires_grad = True
        optimizer = get_optimizer(model, LR_FINETUNE)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - FREEZE_EPOCHS)

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    val_loss,   val_acc   = evaluate(model, val_loader)
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    # ── Save best & early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        no_improve = 0
        flag = "  ← best"
    else:
        no_improve += 1
        flag = f"  (no improve {no_improve}/{PATIENCE})"

    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:.3f}{flag}")

    if no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch}. Best val loss: {best_val_loss:.4f}")
        break

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")

## 10. Training Curves

In [ ]:
epochs_ran = len(history["train_loss"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["train_loss"], label="Train Loss")
axes[0].plot(history["val_loss"],   label="Val Loss")
axes[0].axvline(FREEZE_EPOCHS, color="gray", linestyle="--", label="Unfreeze backbone")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="Train Acc")
axes[1].plot(history["val_acc"],   label="Val Acc")
axes[1].axvline(FREEZE_EPOCHS, color="gray", linestyle="--", label="Unfreeze backbone")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle(f"Training Curves (stopped at epoch {epochs_ran})", fontsize=13)
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## 11. Test Set Evaluation

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

overall_acc = (all_preds == all_labels).mean() * 100
macro_f1    = f1_score(all_labels, all_preds, average="macro")

print(f"Overall Accuracy : {overall_acc:.2f}%")
print(f"Macro F1 Score   : {macro_f1:.4f}")
print()
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(all_labels, all_preds, target_names=classes))

## 12. Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(14, 11))
sns.heatmap(cm_norm, annot=True, fmt=".2f",
            xticklabels=classes, yticklabels=classes,
            cmap="Blues", linewidths=0.5)
plt.title("Normalized Confusion Matrix — Solar Panel Fault Classification")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

## 13. Per-Class F1 Score

In [ ]:
per_class_f1 = f1_score(all_labels, all_preds, average=None)

plt.figure(figsize=(12, 4))
plt.bar(classes, per_class_f1, color="steelblue", edgecolor="black")
plt.axhline(macro_f1, color="red", linestyle="--", label=f"Macro F1 = {macro_f1:.3f}")
plt.title("Per-Class F1 Score")
plt.ylabel("F1 Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig("per_class_f1.png", dpi=150)
plt.show()

## 14. Improvement Strategy — Ablation Study

| Experiment | Dropout | Weight Decay | Sampler | Early Stop | Expected Val Acc |
|---|---|---|---|---|---|
| v1 Baseline | 0.3 | 1e-4 | Yes | No | ~33% (overfit) |
| v2 Fixed | 0.5 | 1e-3 | No | Yes | ~60–75% |

Fill in your actual results below after running both versions.


In [ ]:
ablation_results = {
    "Experiment":    ["v1 — Baseline (overfit)", "v2 — Overfitting Fixed"],
    "Val Acc":       [0.327, 0.0],     # fill in v2 result
    "Macro F1":      [0.0,   0.0],     # fill in both
    "Hot-Spot F1":   [0.0,   0.0],
    "No-Anomaly F1": [0.0,   0.0],
}

ablation_df = pd.DataFrame(ablation_results)
print(ablation_df.to_string(index=False))

## 15. Grad-CAM — Fault Localization
Visualizes *where* in the image the model found the fault. Install with: `pip install grad-cam`

In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    import math

    target_layers = [model.features[-1]]
    cam = GradCAM(model=model, target_layers=target_layers)

    n_show = 12
    indices = random.sample(range(len(test_dataset)), n_show)
    cols = 4
    rows = math.ceil(n_show / cols)

    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 6, rows * 3.5))

    for i, idx in enumerate(indices):
        img_tensor, true_label = test_dataset[idx]
        input_tensor = img_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(input_tensor).argmax(1).item()

        # Grad-CAM
        targets = [ClassifierOutputTarget(pred)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0]

        # Denormalize for display
        mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
        std  = torch.tensor(IMAGENET_STD).view(3,1,1)
        img_np = (img_tensor * std + mean).clamp(0,1).permute(1,2,0).numpy()

        cam_image = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)

        row, col = divmod(i, cols)
        color = "green" if pred == true_label else "red"

        axes[row, col*2].imshow(img_np)
        axes[row, col*2].set_title(f"True: {idx2label[true_label]}", fontsize=8)
        axes[row, col*2].axis("off")

        axes[row, col*2+1].imshow(cam_image)
        axes[row, col*2+1].set_title(f"Pred: {idx2label[pred]}", fontsize=8, color=color)
        axes[row, col*2+1].axis("off")

    plt.suptitle("Original vs Grad-CAM Heatmap (green=correct, red=wrong)", fontsize=12)
    plt.tight_layout()
    plt.savefig("gradcam_results.png", dpi=150)
    plt.show()

except ImportError:
    print("Install grad-cam first: pip install grad-cam")

## 16. Predict a Single Image

In [ ]:
def predict(image_path):
    img = Image.open(image_path).convert("RGB")
    tensor = eval_transforms(img).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs   = torch.softmax(outputs, dim=1)[0]
        conf, pred = probs.max(0)

    print(f"Predicted fault : {idx2label[pred.item()]}")
    print(f"Confidence      : {conf.item():.1%}")
    print()
    print("All class probabilities:")
    for i, (cls, p) in enumerate(zip(classes, probs.tolist())):
        bar = "█" * int(p * 40)
        print(f"  {cls:<20} {p:.3f}  {bar}")

    return idx2label[pred.item()], conf.item()

# Usage — change path to any image
# predict(r"C:/path/to/your/images/13357.jpg")